<p style="text-align:center">
    <a href="https://skills.network" target="_blank">
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/assets/logos/SN_web_lightmode.png" width="200" alt="Skills Network Logo">
    </a>
</p>


# **Análisis de Ubicaciones de Sitios de Lanzamiento con Folium**[cite: 5]


Tiempo estimado necesario: **40** minutos[cite: 5]


La tasa de éxito del lanzamiento puede depender de muchos factores, como la masa de la carga útil, el tipo de órbita, etc.[cite: 5] También puede depender de la ubicación y las proximidades de un sitio de lanzamiento, es decir, la posición inicial de las trayectorias de los cohetes.[cite: 5] Encontrar una ubicación óptima para construir un sitio de lanzamiento ciertamente involucra muchos factores y, con suerte, podríamos descubrir algunos de ellos analizando las ubicaciones de los sitios de lanzamiento existentes.[cite: 5]


En los laboratorios anteriores de análisis exploratorio de datos, ha visualizado el conjunto de datos de lanzamientos de SpaceX utilizando `matplotlib` y `seaborn` y ha descubierto algunas correlaciones preliminares entre el sitio de lanzamiento y las tasas de éxito.[cite: 5] En este laboratorio, realizará un análisis visual más interactivo utilizando `Folium`.[cite: 5]


## Objetivos[cite: 5]


Este laboratorio contiene las siguientes tareas:[cite: 5]
- **TAREA 1:** Marcar todos los sitios de lanzamiento en un mapa[cite: 5]
- **TAREA 2:** Marcar los lanzamientos exitosos/fallidos para cada sitio en el mapa[cite: 5]
- **TAREA 3:** Calcular las distancias entre un sitio de lanzamiento y sus proximidades[cite: 5]

Después de completar las tareas anteriores, debería poder encontrar algunos patrones geográficos sobre los sitios de lanzamiento.[cite: 5]


Primero importemos los paquetes de Python necesarios para este laboratorio:[cite: 5]


In [1]:
!pip3 install folium
!pip3 install wget
!pip3 install pandas

  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Created wheel for wget: filename=wget-3.2-py3-none-any.whl size=9711 sha256=df05ddef8df1b1334959fa484cc91114437a2510ef8484734e851df6ce663328
  Stored in directory: c:\users\mauro\appdata\local\pip\cache\wheels\8a\b8\04\0c88fb22489b0c049bee4e977c5689c7fe597d6c4b0e7d0b6a
Successfully built wget


In [3]:
import folium
import wget
import pandas as pd

In [4]:
# Importar el complemento MarkerCluster de folium[cite: 5]
from folium.plugins import MarkerCluster
# Importar el complemento MousePosition de folium[cite: 5]
from folium.plugins import MousePosition
# Importar el complemento DivIcon de folium[cite: 5]
from folium.features import DivIcon

Si necesita refrescar su memoria sobre folium, puede descargar y consultar este laboratorio anterior sobre folium:[cite: 5]


[Generando Mapas con Python](https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/DV0101EN-3-5-1-Generating-Maps-in-Python-py-v2.0.ipynb)[cite: 5]


## Tarea 1: Marcar todos los sitios de lanzamiento en un mapa[cite: 5]


Primero, intentemos agregar la ubicación de cada sitio en un mapa usando las coordenadas de latitud y longitud del sitio[cite: 5]


El siguiente conjunto de datos con el nombre `spacex_launch_geo.csv` es un conjunto de datos aumentado con latitud y longitud agregadas para cada sitio.[cite: 5]


In [5]:
# Descargar y leer el `spacex_launch_geo.csv`
spacex_csv_file = wget.download('https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/datasets/spacex_launch_geo.csv')
spacex_df=pd.read_csv(spacex_csv_file)

Ahora, puede echar un vistazo a cuáles son las coordenadas de cada sitio.[cite: 5]


In [6]:
# Seleccionar sub-columnas relevantes: `Launch Site`, `Lat(Latitude)`, `Long(Longitude)`, `class`
spacex_df = spacex_df[['Launch Site', 'Lat', 'Long', 'class']]
launch_sites_df = spacex_df.groupby(['Launch Site'], as_index=False).first()
launch_sites_df = launch_sites_df[['Launch Site', 'Lat', 'Long']]
launch_sites_df

,Launch Site,Lat,Long
0,CCAFS LC-40,28.562302,-80.577356
1,CCAFS SLC-40,28.563197,-80.576820
2,KSC LC-39A,28.573255,-80.646895
3,VAFB SLC-4E,34.632834,-120.610745


Las coordenadas anteriores son solo números simples que no pueden brindarle información intuitiva sobre dónde se encuentran esos sitios de lanzamiento.[cite: 5] Si es muy bueno en geografía, puede interpretar esos números directamente en su mente. Si no, también está bien. Visualicemos esas ubicaciones fijándolas en un mapa.[cite: 5]


Primero necesitamos crear un objeto `Map` de folium, con una ubicación central inicial en el Centro Espacial Johnson de la NASA en Houston, Texas.[cite: 5]


In [7]:
# La ubicación de inicio es el Centro Espacial Johnson de la NASA
nasa_coordinate = [29.559684888503615, -95.0830971930759]
site_map = folium.Map(location=nasa_coordinate, zoom_start=10)

Podríamos usar `folium.Circle` para agregar un área circular resaltada con una etiqueta de texto en una coordenada específica. Por ejemplo,[cite: 5]


In [8]:
# Crear un círculo azul en las coordenadas del Centro Espacial Johnson de la NASA con una etiqueta emergente que muestre su nombre
circle = folium.Circle(nasa_coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup('NASA Johnson Space Center'))
# Crear un marcador en las coordenadas del Centro Espacial Johnson de la NASA con un icono que muestre su nombre
marker = folium.map.Marker(
    nasa_coordinate,
    # Crear un icono como una etiqueta de texto
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'NASA JSC',
        )
    )
site_map.add_child(circle)
site_map.add_child(marker)

y debería encontrar un pequeño círculo amarillo cerca de la ciudad de Houston y puede acercar el zoom para ver un círculo más grande.[cite: 5]


Ahora, agreguemos un círculo para cada sitio de lanzamiento en el data frame `launch_sites`[cite: 5]


_TODO:_ Cree y agregue `folium.Circle` y `folium.Marker` para cada sitio de lanzamiento en el mapa de sitios[cite: 5]


Un ejemplo de folium.Circle:[cite: 5]


`folium.Circle(coordinate, radius=1000, color='#000000', fill=True).add_child(folium.Popup(...))`


Un ejemplo de folium.Marker:[cite: 5]


`folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % 'label', ))`


In [9]:
# Inicializar el mapa
site_map = folium.Map(location=nasa_coordinate, zoom_start=5)
# Para cada sitio de lanzamiento, agregue un objeto Circle según los valores de sus coordenadas (Lat, Long). Además, agregue el nombre del sitio de lanzamiento como una etiqueta emergente[cite: 5]
for index, row in launch_sites_df.iterrows():
    coordinate = [row['Lat'], row['Long']]
    folium.Circle(coordinate, radius=1000, color='#d35400', fill=True).add_child(folium.Popup(row['Launch Site'])).add_to(site_map)
    folium.map.Marker(coordinate, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % row['Launch Site'], )).add_to(site_map)
site_map

El mapa generado con los sitios de lanzamiento marcados debería ser similar al siguiente:[cite: 5]


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_markers.png">
</center>


Ahora, puede explorar el mapa acercando y alejando las áreas marcadas e intentar responder las siguientes preguntas:[cite: 5]
- ¿Todos los sitios de lanzamiento están cerca de la línea del Ecuador?[cite: 5]
- ¿Todos los sitios de lanzamiento están muy cerca de la costa?[cite: 5]

También intente explicar sus hallazgos.[cite: 5]


# Tarea 2: Marcar los lanzamientos exitosos/fallidos para cada sitio en el mapa[cite: 5]


A continuación, intentemos mejorar el mapa agregando los resultados de los lanzamientos para cada sitio y veamos qué sitios tienen tasas de éxito altas.[cite: 5]
Recuerde que el data frame spacex_df tiene registros de lanzamiento detallados y la columna `class` indica si este lanzamiento fue exitoso o no.[cite: 5]


In [10]:
spacex_df.tail(10)

,Launch Site,Lat,Long,class
46,KSC LC-39A,28.573255,-80.646895,1
47,KSC LC-39A,28.573255,-80.646895,1
48,KSC LC-39A,28.573255,-80.646895,1
49,CCAFS SLC-40,28.563197,-80.576820,1
50,CCAFS SLC-40,28.563197,-80.576820,1
51,CCAFS SLC-40,28.563197,-80.576820,0
52,CCAFS SLC-40,28.563197,-80.576820,0
53,CCAFS SLC-40,28.563197,-80.576820,0
54,CCAFS SLC-40,28.563197,-80.576820,1
55,CCAFS SLC-40,28.563197,-80.576820,0


A continuación, creemos marcadores para todos los registros de lanzamiento.[cite: 5]
Si un lanzamiento fue exitoso `(class=1)`, entonces usamos un marcador verde y si un lanzamiento falló, usamos un marcador rojo `(class=0)`[cite: 5]


Tenga en cuenta que un lanzamiento solo ocurre en uno de los cuatro sitios de lanzamiento, lo que significa que muchos registros de lanzamiento tendrán exactamente la misma coordenada.[cite: 5] Los grupos de marcadores (Marker clusters) pueden ser una buena forma de simplificar un mapa que contiene muchos marcadores con la misma coordenada.[cite: 5]


Primero creemos un objeto `MarkerCluster`[cite: 5]


In [11]:
marker_cluster = MarkerCluster()


_TODO:_ Cree una nueva columna en el dataframe `launch_sites` llamada `marker_color` para almacenar los colores de los marcadores según el valor de la `class`[cite: 5]


In [12]:

# Aplique una función para verificar el valor de la columna `class`[cite: 5]
# Si class=1, el valor de marker_color será green (verde)[cite: 5]
# Si class=0, el valor de marker_color será red (rojo)[cite: 5]


In [13]:
# Función para asignar color al resultado del lanzamiento
def assign_marker_color(launch_outcome):
    if launch_outcome == 1:
        return 'green'
    else:
        return 'red'
    
spacex_df['marker_color'] = spacex_df['class'].apply(assign_marker_color)
spacex_df.tail(10)

,Launch Site,Lat,Long,class,marker_color
46,KSC LC-39A,28.573255,-80.646895,1,green
47,KSC LC-39A,28.573255,-80.646895,1,green
48,KSC LC-39A,28.573255,-80.646895,1,green
49,CCAFS SLC-40,28.563197,-80.576820,1,green
50,CCAFS SLC-40,28.563197,-80.576820,1,green
51,CCAFS SLC-40,28.563197,-80.576820,0,red
52,CCAFS SLC-40,28.563197,-80.576820,0,red
53,CCAFS SLC-40,28.563197,-80.576820,0,red
54,CCAFS SLC-40,28.563197,-80.576820,1,green
55,CCAFS SLC-40,28.563197,-80.576820,0,red


_TODO:_ Para cada resultado de lanzamiento en el data frame `spacex_df`, agregue un `folium.Marker` a `marker_cluster`[cite: 5]


In [14]:
# Agregue marker_cluster al site_map actual
site_map.add_child(marker_cluster)

# por cada fila en el data frame spacex_df[cite: 5]
# cree un objeto Marker con su coordenada[cite: 5]
# y personalice la propiedad icon del Marker para indicar si este lanzamiento fue exitoso o fallido,[cite: 5]
# por ejemplo, icon=folium.Icon(color='white', icon_color=row['marker_color'][cite: 5]
for index, record in spacex_df.iterrows():
    # TODO: Cree y agregue un grupo de Marcadores (Marker cluster) al mapa de sitios[cite: 5]
    marker = folium.Marker(
        [record['Lat'], record['Long']],
        icon=folium.Icon(color='white', icon_color=record['marker_color'])
    )
    marker_cluster.add_child(marker)

site_map

Su mapa actualizado puede parecerse a las siguientes capturas de pantalla:[cite: 5]


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster.png">
</center>


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_cluster_zoomed.png">
</center>


A partir de los marcadores etiquetados con colores en los grupos de marcadores, debería poder identificar fácilmente qué sitios de lanzamiento tienen tasas de éxito relativamente altas.[cite: 5]


# TAREA 3: Calcular las distancias entre un sitio de lanzamiento y sus proximidades[cite: 5]


A continuación, debemos explorar y analizar las proximidades de los sitios de lanzamiento.[cite: 5]


Primero agreguemos una posición de ratón (`MousePosition`) en el mapa para obtener las coordenadas al pasar el ratón sobre un punto en el mapa.[cite: 5] De esta manera, mientras explora el mapa, puede encontrar fácilmente las coordenadas de cualquier punto de interés (como el ferrocarril)[cite: 5]


In [15]:
# Agregue Mouse Position para obtener la coordenada (Lat, Long) para el paso del ratón en el mapa
formatter = "function(num) {return L.Util.formatNum(num, 5);};"
mouse_position = MousePosition(
    position='topright',
    separator=' Long: ',
    empty_string='NaN',
    lng_first=False,
    num_digits=20,
    prefix='Lat:',
    lat_formatter=formatter,
    lng_formatter=formatter,
)

site_map.add_child(mouse_position)
site_map

Ahora acerque el zoom a un sitio de lanzamiento y explore su proximidad para ver si puede encontrar fácilmente algún ferrocarril, autopista, costa, etc.[cite: 5] Mueva su ratón a estos puntos y anote sus coordenadas (mostradas en la parte superior izquierda) para conocer la distancia al sitio de lanzamiento.[cite: 5]


Puede calcular la distancia entre dos puntos en el mapa en función de sus valores `Lat` y `Long` utilizando el siguiente método:[cite: 5]


In [16]:
from math import sin, cos, sqrt, atan2, radians

def calculate_distance(lat1, lon1, lat2, lon2):
    # radio aproximado de la tierra en km
    R = 6373.0

    lat1 = radians(lat1)
    lon1 = radians(lon1)
    lat2 = radians(lat2)
    lon2 = radians(lon2)

    dlon = lon2 - lon1
    dlat = lat2 - lat1

    a = sin(dlat / 2)**2 + cos(lat1) * cos(lat2) * sin(dlon / 2)**2
    c = 2 * atan2(sqrt(a), sqrt(1 - a))

    distance = R * c
    return distance

_TODO:_ Marque un punto en la línea costera más cercana usando MousePosition y calcule la distancia entre el punto de la costa y el sitio de lanzamiento.[cite: 5]


In [17]:
# encuentre la coordenada de la costa más cercana[cite: 5]
# por ejemplo: Lat: 28.56367  Lon: -80.57163[cite: 5]
# distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)[cite: 5]
launch_site_lat = 28.563197
launch_site_lon = -80.576820
coastline_lat = 28.56334
coastline_lon = -80.56799
distance_coastline = calculate_distance(launch_site_lat, launch_site_lon, coastline_lat, coastline_lon)

_TODO:_ Después de obtener su coordenada, cree un `folium.Marker` para mostrar la distancia[cite: 5]


In [18]:
# Cree y agregue un folium.Marker en el punto de la costa más cercano seleccionado en el mapa[cite: 5]
# Muestre la distancia entre el punto de la costa y el sitio de lanzamiento usando la propiedad icon[cite: 5]
# por ejemplo
distance_marker = folium.Marker(
    [coastline_lat, coastline_lon],
    icon=DivIcon(
        icon_size=(20,20),
        icon_anchor=(0,0),
        html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_coastline),
        )
    )
site_map.add_child(distance_marker)

_TODO:_ Dibuje una `PolyLine` entre un sitio de lanzamiento y el punto de la costa seleccionado[cite: 5]


In [19]:
# Cree un objeto `folium.PolyLine` usando las coordenadas de la costa y la coordenada del sitio de lanzamiento[cite: 5]
coordinates = [[launch_site_lat, launch_site_lon], [coastline_lat, coastline_lon]]
lines=folium.PolyLine(locations=coordinates, weight=1)
site_map.add_child(lines)

Su mapa actualizado con la línea de distancia debería verse como la siguiente captura de pantalla:[cite: 5]


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/launch_site_marker_distance.png">
</center>


_TODO:_ Del mismo modo, puede dibujar una línea entre un sitio de lanzamiento y la ciudad, ferrocarril, carretera, etc. más cercanos.[cite: 5] Primero debe usar `MousePosition` para encontrar sus coordenadas en el mapa[cite: 5]


El símbolo de un mapa ferroviario puede tener este aspecto:[cite: 5]


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/railway.png">
</center>


El símbolo de un mapa de carreteras puede tener este aspecto:[cite: 5]


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/highway.png">
</center>


El símbolo del mapa de una ciudad puede tener este aspecto:[cite: 5]


<center>
    <img src="https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/IBM-DS0321EN-SkillsNetwork/labs/module_3/images/city.png">
</center>


In [20]:
# Cree un marcador con la distancia a la ciudad, ferrocarril, carretera más cercana, etc.[cite: 5]
# Dibuje una línea entre el marcador y el sitio de lanzamiento[cite: 5]
closest_highway = 28.56335, -80.57085
closest_railroad = 28.57206, -80.58525
closest_city = 28.10473, -80.64531

distance_highway = calculate_distance(launch_site_lat, launch_site_lon, closest_highway[0], closest_highway[1])
distance_railroad = calculate_distance(launch_site_lat, launch_site_lon, closest_railroad[0], closest_railroad[1])
distance_city = calculate_distance(launch_site_lat, launch_site_lon, closest_city[0], closest_city[1])

In [21]:
# Marcador y línea para la autopista (Highway)
highway_marker = folium.Marker(closest_highway, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_highway),))
site_map.add_child(highway_marker)
site_map.add_child(folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], closest_highway], weight=1))

# Marcador y línea para el ferrocarril (Railroad)
railroad_marker = folium.Marker(closest_railroad, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_railroad),))
site_map.add_child(railroad_marker)
site_map.add_child(folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], closest_railroad], weight=1))

# Marcador y línea para la ciudad (City)
city_marker = folium.Marker(closest_city, icon=DivIcon(icon_size=(20,20),icon_anchor=(0,0), html='<div style="font-size: 12; color:#d35400;"><b>%s</b></div>' % "{:10.2f} KM".format(distance_city),))
site_map.add_child(city_marker)
site_map.add_child(folium.PolyLine(locations=[[launch_site_lat, launch_site_lon], closest_city], weight=1))

In [22]:
site_map

Después de trazar las líneas de distancia hacia las proximidades, puede responder las siguientes preguntas fácilmente:[cite: 5]
- ¿Los sitios de lanzamiento están muy cerca de las vías del tren?[cite: 5]
- ¿Los sitios de lanzamiento están muy cerca de las autopistas?[cite: 5]
- ¿Los sitios de lanzamiento están muy cerca de la costa?[cite: 5]
- ¿Los sitios de lanzamiento se mantienen a cierta distancia de las ciudades?[cite: 5]

También intente explicar sus hallazgos.[cite: 5]


# Próximos Pasos:[cite: 5]

Ahora ha descubierto mucha información interesante relacionada con la ubicación de los sitios de lanzamiento utilizando folium, de una manera muy interactiva.[cite: 5] A continuación, deberá crear un panel (dashboard) utilizando Plotly Dash sobre los registros detallados de los lanzamientos.[cite: 5]


## Autores[cite: 5]


[Yan Luo](https://www.linkedin.com/in/yan-luo-96288783/)[cite: 5]


### Otros Contribuyentes[cite: 5]


Joseph Santarcangelo[cite: 5]


## Registro de Cambios[cite: 5]


|Fecha (AAAA-MM-DD)|Versión|Cambiado Por|Descripción del Cambio|
|-|-|-|-|
|2021-05-26|1.0|Yan|Versión inicial creada|


Derechos de autor © 2021 IBM Corporation. Todos los derechos reservados.[cite: 5]
